In [ ]:
import pandas as pd
import numpy as np
import pathlib
import sys

PROJECT_ROOT = pathlib.Path.cwd().resolve()
if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PATH = PROJECT_ROOT / "data"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# one of 2 datasets

In [ ]:

behavior=DATA_PATH / "shopping_behavior_updated.csv"
cons_habs=pd.read_csv(behavior)
mymap={'Yes':1,'No':0}
for col in ['Subscription Status', 'Discount Applied']:
    cons_habs[col]=cons_habs[col].map(mymap).astype('object')
def freq_factor(x):
    if x=='Every 3 Months': return 365/4
    if x=='Annually': return 365/1
    if x=='Quarterly': return 365/4
    if x=='Monthly': return 365/12
    if x=='Bi-Weekly': return 7/2
    if x=='Fortnightly': return 14
    if x=='Weekly': return 7
cons_habs['Total Days of Patronage']=(cons_habs['Frequency of Purchases'].map(freq_factor)*cons_habs['Previous Purchases']).astype(int)
cons_habs=cons_habs.drop(columns='Customer ID')
cons_habs.head()

In [ ]:
cons_habs.dtypes

# import 2 of 2 datasets

In [ ]:
coff_by_reg=DATA_PATH / "US_FL_IL_2020_to_2025_Weekly_googtrendcoffeesearch.csv"
coffee=pd.read_csv(coff_by_reg)
coffee=coffee[coffee.columns[1:]]
coffee.head()

In [ ]:
coffee.corr()

# import binning tools

In [ ]:
from data_analysis_utils.BinnerClass import Bin
bin=Bin()

# a look at coffee

In [ ]:
coffee.dtypes

In [ ]:
for col in coffee.columns:
    print(f"{col}: has {coffee[col].nunique()} unique integers and {coffee[col].shape[0]} rows.")

# ========================================================================================================  
># THE FOLLOWING CELLS DEAL WITH FINDING MINIMUM NUMBER OF BINS THAT RETAIN STATISTICAL SIGNIFICANCE  

# Target n  variables against the rest

In [ ]:
numeric_columns=None 
categoric_columns=None
categoric_target=None
numeric_target=['Illinois','Florida']  ###  TARGET THESE VARIABLES


bin.relational_binner(coffee,   # a pandas dataframe
                    numnum_meth_alpha_above=('pearson',0.9,True),    # Numeric to numeric instructions: a tuple with (test method, threshold, keep >= threshold): (str,float,bool)
                    numcat_meth_alpha_above=('kruskal',0.05,False),    # Categoric to numeric instructions: a tuple with (test method, threshold, keep >= threshold): (str,float,bool)
                    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     
                    categoric_columns=categoric_columns,    
                    numeric_target=numeric_target,      
                    categoric_target=categoric_target  )  
bin.numeric_target_column_minimums

# The entire dataset in one shot 

In [ ]:
numeric_columns=None 
categoric_columns=None
categoric_target=None
numeric_target=None

bin.relational_binner(coffee,  
                    numnum_meth_alpha_above=('pearson',0.9,True),    
                    numcat_meth_alpha_above=None,   
                    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     
                    categoric_columns=categoric_columns,    
                    numeric_target=numeric_target,      
                    categoric_target=categoric_target  )
in_depth=bin.numeric_feature_col_thresholds
bin.numeric_target_column_minimums

# A close look at bin metrics for Illinois  
This explores binning strategies for all columns Illinois is related to based on the pearson correlation method used above.   
Near the end of this .ipynb are plots that cary this idea across all outputs

In [ ]:
in_depth['Illinois']

# One-shot approach using spearman coefficient, pearson coefficient, and kendall's rank  

In [ ]:
numeric_columns=None 
categoric_columns=None
categoric_target=None
numeric_target=None
bin.relational_binner(coffee,   numnum_meth_alpha_above=('pearson',0.9,True),   numcat_meth_alpha_above=('kruskal',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"pearson: {bin.numeric_target_column_minimums}")
bin.relational_binner(coffee,  numnum_meth_alpha_above=('spearman',0.9,True),    numcat_meth_alpha_above=('kruskal',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,  numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"spearman: {bin.numeric_target_column_minimums}")
bin.relational_binner(coffee,  numnum_meth_alpha_above=('kendall',0.7,True),   numcat_meth_alpha_above=('kruskal',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,   categoric_columns=categoric_columns,     numeric_target=numeric_target,     categoric_target=categoric_target  )
print(f"kendall: {bin.numeric_target_column_minimums}")

# One-shot approach using student's t-test, welch's t-test  
### Once with bins above p-value, and once below p-value  

In [ ]:
bin.relational_binner(coffee,numnum_meth_alpha_above=('welch',0.05,False),   numcat_meth_alpha_above=None,    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"welch reject null: {bin.numeric_target_column_minimums}")
#print(bin.numeric_feature_col_thresholds)
bin.relational_binner(coffee,numnum_meth_alpha_above=('welch',0.05,True),   numcat_meth_alpha_above=None,    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"welch fail to reject null: {bin.numeric_target_column_minimums}")
#print(bin.numeric_feature_col_thresholds)
bin.relational_binner(coffee,numnum_meth_alpha_above=('student',0.05,False),   numcat_meth_alpha_above=None,    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"student reject null: {bin.numeric_target_column_minimums}")
bin.relational_binner(coffee,numnum_meth_alpha_above=('student',0.05,True),   numcat_meth_alpha_above=None,    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"student fail to reject null: {bin.numeric_target_column_minimums}")

# A look at consumer habits dataset

# Binning numeric columns based on a targeted categorical column: "Gender"

In [ ]:
numeric_columns=None
categoric_columns=None
categoric_target='Gender'
numeric_target=None 
bin.relational_binner(cons_habs,numnum_meth_alpha_above=None,   numcat_meth_alpha_above=('kruskal',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis reject null: {bin.numeric_target_column_minimums}")
#print(bin.numeric_feature_col_thresholds)
bin.relational_binner(cons_habs,numnum_meth_alpha_above=None,   numcat_meth_alpha_above=('kruskal',0.05,True),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis null: {bin.numeric_target_column_minimums}")
#print(bin.numeric_feature_col_thresholds)
bin.relational_binner(cons_habs,numnum_meth_alpha_above=None,   numcat_meth_alpha_above=('anova',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"ANOVA reject null: {bin.numeric_target_column_minimums}")
bin.relational_binner(cons_habs,numnum_meth_alpha_above=None,   numcat_meth_alpha_above=('anova',0.05,True),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"ANOVA null: {bin.numeric_target_column_minimums}")

# Same categorical target, "Gender", but add a numeric target, "Review Rating".    
The function will look at all numeric columns statisticaly within the threshold with "Gender", and it will look at "Review Rating" for all the columns it is within the threshold with

In [ ]:
numeric_columns=None
categoric_columns=None
categoric_target='Gender'
numeric_target='Review Rating' 
bin.relational_binner(cons_habs,numnum_meth_alpha_above=('welch',0.05,False),   numcat_meth_alpha_above=('kruskal',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis reject null & welch's T-test reject null: {bin.numeric_target_column_minimums}")
bin.relational_binner(cons_habs,numnum_meth_alpha_above=('welch',0.05,True),   numcat_meth_alpha_above=('kruskal',0.05,True),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis null & welch's T-test null: {bin.numeric_target_column_minimums}")

# Manual categorical and numeric input

In [ ]:
numeric_columns=['Total Days of Patronage', 'Review Rating', 'Previous Purchases','Age']  
categoric_columns=['Shipping Type','Gender', 'Item Purchased', 'Category']
categoric_target=None 
numeric_target=None
bin.relational_binner(cons_habs,numnum_meth_alpha_above=('welch',0.05,False),   numcat_meth_alpha_above=('kruskal',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis reject null & welch's T-test reject null: {bin.numeric_target_column_minimums}")
bin.relational_binner(cons_habs,numnum_meth_alpha_above=('welch',0.05,True),   numcat_meth_alpha_above=('kruskal',0.05,True),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis null & welch's T-test null: {bin.numeric_target_column_minimums}")

# One-shot approach with no targets

In [ ]:
numeric_columns=None
categoric_columns=None
categoric_target=None
numeric_target=None
bin.relational_binner(cons_habs,numnum_meth_alpha_above=('welch',0.05,False),   numcat_meth_alpha_above=('kruskal',0.05,False),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis reject null & welch's T-test reject null: {bin.numeric_target_column_minimums}")
reject_breakdown_metrics=bin.numeric_feature_col_thresholds
bin.relational_binner(cons_habs,numnum_meth_alpha_above=('welch',0.05,True),   numcat_meth_alpha_above=('kruskal',0.05,True),    original_value_count_threashold=5,  
                    numeric_columns=numeric_columns,     categoric_columns=categoric_columns,    numeric_target=numeric_target,      categoric_target=categoric_target  )
print(f"kruskal_wallis null & welch's T-test null: {bin.numeric_target_column_minimums}")
null_breakdown_metrics=bin.numeric_feature_col_thresholds

# A look at the column-to-column breakdown of fail-to-reject and of reject bin thresholds

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
def plot_min_within_threshold(binning_metrics):
    for outer_key, inner_dict in binning_metrics.items():
        df = pd.DataFrame.from_dict(
            {
                k: v['min_within_threshold']
                for k, v in inner_dict.items()
            },
            orient="index",
            columns=["min_within_threshold"]
        ).reset_index().rename(columns={"index": "feature"})
        df = df.sort_values("min_within_threshold", ascending=False).reset_index(drop=True)

        plt.style.use('seaborn-v0_8-darkgrid')
        plt.figure(figsize=(20,4))
        sns.barplot(
            data=df,
            x="feature",
            y="min_within_threshold"
        )
        fontsize=15
        plt.title(f"\n{outer_key}\n",fontsize=fontsize+5)
        plt.xlabel("Feature",fontsize=fontsize-5)
        plt.ylabel("Min Bins Within Threshold",fontsize=fontsize-5)
        plt.xticks(rotation=20, ha="right",fontsize=fontsize)
        plt.tight_layout()
        #plt.grid()
        plt.show()


# Bins that maintain rejected null thresholds  

In [ ]:
plot_min_within_threshold(reject_breakdown_metrics)

# Bins that maintain fail to reject null thresholds

In [ ]:
plot_min_within_threshold(null_breakdown_metrics)